# Browse generations

Reads a `*_responses.jsonl` file line by line and keeps only rows matching `FILTERS`, so large files never need to be
opened in the editor. Set `PATH`, `FILTERS` and `N` in the next cell, then run all cells.

In [ ]:
import json
import textwrap
from collections import Counter
from pathlib import Path

GENERATIONS = Path.cwd()  # this notebook's folder: generations/
PATH = GENERATIONS / "Llama-3.1-8B-Instruct/training/synthetic_dataset/synthetic_trait_emotion_fact_opinion/synthetic_trait_emotion_fact_opinion_contrastive_responses.jsonl"

# Keep rows whose field equals the value, or is in the list. Empty dict keeps every row.
# Fields include pair_index, cell, pair_type, polarity, label, prompt_id, scenario_id, fact, domain,
# trait_type, emotion_type, truncated, openrouter_provider.
FILTERS = {"pair_index": 0, "fact": "false"}
N = 10  # rows to print

In [ ]:
def matches(row, filters):
    for key, want in filters.items():
        if key not in row:
            raise KeyError(f"{key!r} is not a field; fields are {sorted(row)}")
        if row[key] not in (want if isinstance(want, list) else [want]):
            return False
    return True


n_total = 0
rows = []
with open(PATH, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            n_total += 1
            row = json.loads(line)
            if matches(row, FILTERS):
                rows.append(row)
print(f"{len(rows)} of {n_total} rows match {FILTERS}")

## Counts in the selection

In [ ]:
for key in ("cell", "polarity", "fact", "truncated", "openrouter_provider"):
    if rows and key in rows[0]:
        print(f"{key}: {dict(Counter(r[key] for r in rows))}")

## Rows

In [ ]:
def show(row, width=110):
    print("=" * width)
    print(" | ".join(f"{k}={row[k]}" for k in ("id", "cell", "polarity", "fact", "truncated", "openrouter_provider") if k in row))
    for m in row["messages"]:
        print(f"\n[{m['role'].upper()}]")
        print(textwrap.fill(m["content"], width))
    print("\n[RESPONSE]")
    for para in row["response"].split("\n"):
        print(textwrap.fill(para, width) if para else "")


for row in rows[:N]:
    show(row)

## One prompt under both sides of a pair

In [ ]:
PROMPT_ID = rows[0]["prompt_id"] if rows else None  # or set a prompt_id, e.g. "s000_trait_emotion_fact_opinion_false"
PAIR_INDEX = FILTERS.get("pair_index", 0)

with open(PATH, encoding="utf-8") as f:
    both = [r for r in map(json.loads, filter(str.strip, f)) if r["prompt_id"] == PROMPT_ID and r.get("pair_index") == PAIR_INDEX]
for row in sorted(both, key=lambda r: r["polarity"], reverse=True):
    show(row)